# Notebook 01 — SimpleMLP 784→8→4→2 on MNIST Even/Odd (digits 0,1,3,4)

**Architecture:** SimpleMLP with hidden dims [8, 4], trained on MNIST digits 0, 1, 3, 4 with even/odd binary labels.

## Structure
1. **Setup** — imports, config, device
2. **Models** — load or train 5 seeds into `data/models/`
3. **Factorization & Inspection** — BFT on seed 0: factor panels, pixel receptive fields, scaffold graphs
4. **Ablation** — 5-seed comparative sweep + causal circuit ablation
5. **New Stimuli via NNLS** — round-trip, ID train, near-OOD (excluded digits), far-OOD (synthetic)

## §0 — Imports & Config

In [1]:
#%matplotlib inline
import sys, os, pickle, copy
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import cosine_distances

from src import (
    SimpleMLP,
    load_experiment, get_transform, get_loaders_from_config,
    collect_layer_inputs, collect_layer_inputs_generic, collect_layer_dicts,
    bft, evaluate,
    build_scaffold_edges, scaffold_loading_from_edges, plot_scaffold_graph,
    assign_factors_to_classes, path_from_child, extract_importance_scores,
    magnitude_scores, run_ablation_sweep, per_class_accuracy,
    select_class_circuit, normalize_scores_per_layer,
    ablation_layer_sweep,
    extract_tree_nodes, extract_factor_tree_nodes,
    extract_fingerprint_matrix, compute_stimulus_similarity,
    project_stimuli_onto_tree, project_onto_bft, compute_factor_activations,
    nodes_at_layer, plot_factor_tree, top_stimuli_factor_activations,
    plot_factor_overview_panel, plot_factor_gallery, plot_input_layer_factors,
    plot_pruning_results, plot_pruning_by_layer_depth, plot_embedding_comparison,
    compute_nmf_stability, plot_nmf_stability_figure,
    compute_k_sensitivity, plot_k_sensitivity_figure,
    save_experiment,
)
from src.training import train_epoch, label_transform_even_odd
from src.data_utils import get_mnist_loaders

matplotlib.rcParams.update({'figure.dpi': 80})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
_nb_dir   = os.path.dirname(os.path.abspath('__file__'))
_repo_dir = os.path.dirname(_nb_dir)

MODEL_ROOT = os.path.join(_repo_dir, 'data', 'models')
CACHE_ROOT = os.path.join(_repo_dir, 'data', 'cache', 'nb01_mlp84')
FIG_DIR    = os.path.join(_repo_dir, 'figs', '01_mlp_8_4_0134')
for d in [MODEL_ROOT, CACHE_ROOT, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Experiment config ──────────────────────────────────────────────────────────
EXP_BASE      = 'mnist_even_odd_mlp_8_4_0134'
N_SEEDS       = 5
N_EPOCHS      = 30

# ── BFT config ────────────────────────────────────────────────────────────────
N_BRANCHES         = [1, 1, 2]
K_MAX_PER_LAYER    = [5, 5, 5]
STIMULUS_THRESHOLD = 0.5
N_VIZ              = 4

# ── Ablation config ───────────────────────────────────────────────────────────
N_CLASSES        = 2
CLASS_NAMES      = {0: 'even', 1: 'odd'}
ABLATION_FRACS   = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
N_RANDOM_REPEATS = 10
PERCENTILES      = [50, 75, 90, 95, 99]
N_PER_CLASS      = 200

# ── NNLS / OOD ────────────────────────────────────────────────────────────────
TOP_N_FACTOR  = 50
IMAGE_SIDE    = 28
N_FAR_OOD     = 300

print(f'MODEL_ROOT: {MODEL_ROOT}')
print(f'CACHE_ROOT: {CACHE_ROOT}')
print(f'FIG_DIR:    {FIG_DIR}')

MODEL_ROOT: /home/jb3879/Factor_Trace/data/models
CACHE_ROOT: /home/jb3879/Factor_Trace/data/cache/nb01_mlp84
FIG_DIR:    /home/jb3879/Factor_Trace/figs/01_mlp_8_4_0134


## §1 — Models: load or train 5 seeds

In [3]:
BASE_CONFIG = {
    'arch': 'SimpleMLP',
    'arch_kwargs': {'input_dim': 784, 'hidden_dims': [8, 4], 'output_dim': 2},
    'dataset': 'MNIST',
    'dataset_kwargs': {'root': '../data/', 'batch_size': 32,
                       'digit_filter': [0, 1, 3, 4]},
    'label_transform': 'even_odd',
    'analysis_layer_indices': [2, 4, 6],
    'n_per_class': 1000,
    'input_side': 28,
}

models_by_seed = {}

for seed in range(N_SEEDS):
    exp_dir = os.path.join(MODEL_ROOT, f'{EXP_BASE}_seed{seed}')
    if os.path.exists(os.path.join(exp_dir, 'weights.pt')):
        m, cfg = load_experiment(exp_dir, DEVICE)
        models_by_seed[seed] = (m, cfg)
        print(f'Seed {seed}: loaded from {exp_dir}')
    else:
        print(f'Seed {seed}: training from scratch...')
        torch.manual_seed(seed)
        np.random.seed(seed)
        m = SimpleMLP(**BASE_CONFIG['arch_kwargs']).to(DEVICE)
        cfg = dict(BASE_CONFIG, description=f'{EXP_BASE} seed {seed}')

        train_loader, test_loader = get_mnist_loaders(
            batch_size=32, root='../data/', digit_filter=[0,1,3,4])
        label_transform = label_transform_even_odd
        optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        for ep in range(N_EPOCHS):
            train_epoch(m, train_loader, optimizer, criterion, DEVICE, label_transform)
        _, acc = evaluate(m, test_loader, criterion, DEVICE, label_transform)
        print(f'  Test acc: {acc:.4f}')

        cfg['description'] = f'{EXP_BASE} seed {seed}'
        save_experiment(m, cfg, exp_dir)
        models_by_seed[seed] = (m, cfg)
        print(f'  Saved to {exp_dir}')

# Canonical model is seed 0
model, config = models_by_seed[0]
train_loader, test_loader = get_loaders_from_config(config)
label_transform = get_transform(config['label_transform'])
_, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss(), DEVICE, label_transform)

linear_indices = model.linear_layer_indices()
LAYER_SIZES    = [model.layers[li].out_features for li in linear_indices]
print(f'\nSeed-0 test accuracy: {test_acc:.4f}')
print(f'Architecture (per-layer neurons): {LAYER_SIZES}')

Seed 0: loaded from /home/jb3879/Factor_Trace/data/models/mnist_even_odd_mlp_8_4_0134_seed0
Seed 1: loaded from /home/jb3879/Factor_Trace/data/models/mnist_even_odd_mlp_8_4_0134_seed1
Seed 2: loaded from /home/jb3879/Factor_Trace/data/models/mnist_even_odd_mlp_8_4_0134_seed2
Seed 3: loaded from /home/jb3879/Factor_Trace/data/models/mnist_even_odd_mlp_8_4_0134_seed3
Seed 4: loaded from /home/jb3879/Factor_Trace/data/models/mnist_even_odd_mlp_8_4_0134_seed4



Seed-0 test accuracy: 0.9934
Architecture (per-layer neurons): [8, 4, 2]


## §2 — Factorization & Inspection (seed 0)

### 2a — Collect layer data and run BFT

In [4]:
from src.data_utils import label_transformed_loader

# Loader yielding even/odd labels, for BFT primary mode (enables validate=True).
val_loader = label_transformed_loader(test_loader, label_transform)

# Collect the SAME samples BFT primary mode uses (loader order, only_correct), so
# downstream per-sample arrays stay aligned with tree_root's img_factors.
_collected  = collect_layer_dicts(model, test_loader, label_transform=label_transform,
                                  device=DEVICE)
all_images   = _collected['images']                              # (N, 1, 28, 28)
all_targets  = _collected['targets']                             # (N,) even/odd labels
all_digits   = _collected['digits']                              # (N,) original digit
layer_inputs = [d['input_fmap'] for d in _collected['layer_data']]  # list[(N, n_in)]
n_samples    = len(all_images)
flat_imgs    = all_images.reshape(n_samples, -1)

print(f'Samples: {n_samples}')
print(f'Class distribution: {dict(zip(*np.unique(all_targets, return_counts=True)))}')
for i, li in enumerate(layer_inputs):
    print(f'  L{i+1} inputs: {li.shape}')

In [ ]:

# Primary mode: pass (model, loader) so BFT collects internally and validate=True works.
tree_root = bft(
    model, val_loader,
    k_max=K_MAX_PER_LAYER,
    n_branches=N_BRANCHES,
    stimulus_threshold=STIMULUS_THRESHOLD,
    weighting='img_selectivity',
    validate=True,
    verbose=1,
    n_jobs=3,
)
print('Validation summary:', tree_root.validation_summary())


def get_nodes_by_depth(root):
    from collections import deque
    by_depth = {}
    queue = deque([(root, 0)])
    while queue:
        node, d = queue.popleft()
        by_depth.setdefault(d, []).append(node)
        for child in node.children:
            queue.append((child, d+1))
    return by_depth

def get_all_paths(root):
    if not root.children: return [[root]]
    return [[root] + p for child in root.children for p in get_all_paths(child)]

nodes_by_depth = get_nodes_by_depth(tree_root.root)
all_paths      = get_all_paths(tree_root.root)
print(f'Tree: {len(nodes_by_depth)} depths, {sum(len(v) for v in nodes_by_depth.values())} nodes, {len(all_paths)} paths')


[BFT] Layer 3/3 'layers.5' (fc)  path=[]
[BFT L3 'layers.5' (fc)]   K=2  t=0.172s
[BFT L3 'layers.5' (fc)]   recon loss-ratio (all factors) = 1.9253  (n_eval=4080)
[BFT] Layer 2/3 'layers.3' (fc)  path=[0]
[BFT L2 'layers.3' (fc)]   K=1  t=1.310s
[BFT L2 'layers.3' (fc)]   recon loss-ratio (all factors) = 2.8224  (n_eval=100)
[BFT] Layer 1/3 'layers.1' (fc)  path=[0, 0]


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/sklearn/decomposition/_nmf.py:2306: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


[BFT L1 'layers.1' (fc)]   K=5  t=855.824s
[BFT L1 'layers.1' (fc)]   recon loss-ratio (all factors) = 0.9463  (n_eval=100)
[BFT] Layer 2/3 'layers.3' (fc)  path=[1]
[BFT L2 'layers.3' (fc)]   K=2  t=1.631s
[BFT L2 'layers.3' (fc)]   recon loss-ratio (all factors) = 1.0175  (n_eval=100)
[BFT] Layer 1/3 'layers.1' (fc)  path=[1, 0]


In [ ]:

# Primary mode: pass (model, loader) so BFT collects internally and validate=True works.
tree_root = bft(
    model, val_loader,
    k_max=K_MAX_PER_LAYER,
    n_branches=N_BRANCHES,
    stimulus_threshold=STIMULUS_THRESHOLD,
    weighting='img_selectivity',
    validate=True,
    verbose=1,
    n_jobs=3,
)
print('Validation summary:', tree_root.validation_summary())


def get_nodes_by_depth(root):
    from collections import deque
    by_depth = {}
    queue = deque([(root, 0)])
    while queue:
        node, d = queue.popleft()
        by_depth.setdefault(d, []).append(node)
        for child in node.children:
            queue.append((child, d+1))
    return by_depth

def get_all_paths(root):
    if not root.children: return [[root]]
    return [[root] + p for child in root.children for p in get_all_paths(child)]

nodes_by_depth = get_nodes_by_depth(tree_root.root)
all_paths      = get_all_paths(tree_root.root)
print(f'Tree: {len(nodes_by_depth)} depths, {sum(len(v) for v in nodes_by_depth.values())} nodes, {len(all_paths)} paths')


### 2b — Per-layer factor visualizations

In [7]:
# ── Plot 2: Input-layer pixel receptive fields ────────────────────────────────
# Shows connection_factors[:, k] for each L1 node reshaped to (n_out, 28, 28)
max_depth = max(nodes_by_depth.keys())
for r in nodes_by_depth[max_depth]:
    path_label = 'F' + '→F'.join(str(f) for f in r.path) if r.path else 'root'
    figs = plot_input_layer_factors(
        r, all_images, arch='fc', image_shape=(IMAGE_SIDE, IMAGE_SIDE)
    )
    for k, fig in enumerate(figs):
        fname = f"pixel_rf_L{r.layer_idx+1}_{path_label.replace('→','-')}_k{k}.pdf"
        fig.savefig(os.path.join(FIG_DIR, fname), bbox_inches='tight')
        plt.show(); plt.close(fig)


### 2d — Full Network Scaffold Graphs

In [ ]:
for path_nodes in all_paths:
    layer_results = list(reversed(path_nodes))  # L1-first order

    edge_matrices, neg_edge_matrices = build_scaffold_edges(
        layer_results[1:], fi='path', fi_seed=layer_results[0], top_pct=0.05,
    )
    loading = scaffold_loading_from_edges(edge_matrices)

    path_label = 'F' + '→F'.join(str(f) for f in path_nodes[-1].path)

    fig = plot_scaffold_graph(loading, edge_matrices, LAYER_SIZES,
                              neg_edge_matrices=neg_edge_matrices)
    fig.suptitle(f'Scaffold Graph [path: {path_label}]', fontsize=11)

    _pl = path_label.replace('→', '-')
    fig.savefig(os.path.join(FIG_DIR, f'scaffold_{_pl}.pdf'), bbox_inches='tight')
    plt.show()


## §3 — Ablation (seed 0)

Layer-depth pruning sweep: for each target class, prune the last 1, 2, …, L layers at
three fraction levels using BFT-top, BFT-bottom, magnitude, and random baselines.
X-axis = how many layers from the output are included in the pruning pool.

In [ ]:
# ── Ablation config ───────────────────────────────────────────────────────────
METHODS          = ['bft_top', 'bft_bottom', 'magnitude', 'random']
ABLATION_FRACS   = [0.10, 0.20, 0.30]
N_RANDOM_REPEATS = 10

# Test loader restricted to training digits
mnist_test_full = datasets.MNIST('../data/', train=False, download=True, transform=ToTensor())
digit_filter    = config['dataset_kwargs']['digit_filter']
mask            = np.isin(np.array(mnist_test_full.targets), digit_filter)
test_loader_abl = DataLoader(Subset(mnist_test_full, np.where(mask)[0]),
                              batch_size=256, shuffle=False)

In [ ]:
# ── Layer-depth ablation sweep (seed 0) ───────────────────────────────────────
# tree_root from §2 has n_branches=[1,1,2] which matches N_CLASSES=2
layer_sweeps = {}
for d in range(N_CLASSES):
    print(f'\nClass {d} ({CLASS_NAMES[d]})...')
    layer_sweeps[d] = ablation_layer_sweep(
        model, tree_root, test_loader_abl,
        target_class=d,
        fractions=ABLATION_FRACS,
        methods=METHODS,
        label_transform=label_transform_even_odd,
        device=DEVICE,
        n_random_repeats=N_RANDOM_REPEATS,
        verbose=1,
    )

In [ ]:
# ── Plot 6: Pruning by layer depth ────────────────────────────────────────────
for d in range(N_CLASSES):
    fig = plot_pruning_by_layer_depth(
        layer_sweeps[d], fractions=ABLATION_FRACS,
        target_class=d, class_names=CLASS_NAMES,
        methods=METHODS,
    )
    fig.savefig(os.path.join(FIG_DIR, f'pruning_layer_depth_class{d}.pdf'), bbox_inches='tight')
    plt.show()
    plt.close(fig)

## §4 — New Stimuli via NNLS

### 4a — BFT factor tree for NNLS (seed 0)

In [ ]:
nnls_root = bft(
    model, val_loader,
    k_max=[10, 1, 2], n_branches=[8, 2, 2],
    stimulus_threshold=0, weighting='img_selectivity', n_jobs=3,
)


nnls_tree_nodes   = extract_tree_nodes(nnls_root)
nnls_factor_nodes = extract_factor_tree_nodes(nnls_root)

# Round-trip test
from sklearn.metrics.pairwise import paired_cosine_distances as _pcd
projected_test_rt = project_stimuli_onto_tree(nnls_root, layer_inputs)
F_orig = extract_fingerprint_matrix(nnls_root, np.arange(n_samples))
F_rt   = extract_fingerprint_matrix(projected_test_rt, np.arange(n_samples))
rt_sims = 1.0 - _pcd(F_orig, F_rt)
print(f'Round-trip cosine sim: mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}  min={rt_sims.min():.4f}')


### 4b — Factor fingerprints & similarity (test set)

In [ ]:
F = extract_fingerprint_matrix(nnls_root, np.arange(n_samples))
print(f'Fingerprint matrix: {F.shape}')

MAX_VIZ = 300
viz_idx     = np.argsort(all_targets)[:MAX_VIZ] if n_samples > MAX_VIZ else np.argsort(all_targets)
S           = compute_stimulus_similarity(F[viz_idx])
targets_viz = all_targets[viz_idx]

# ── Similarity heatmap + intra/inter histogram ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(S, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
fig.colorbar(im, ax=axes[0])
for cl in np.unique(targets_viz):
    for b in np.where(np.diff((targets_viz == cl).astype(int)))[0]:
        axes[0].axhline(b + 0.5, color='k', lw=0.5)
        axes[0].axvline(b + 0.5, color='k', lw=0.5)
axes[0].set(title='Factor fingerprint similarity (sorted by class)',
            xlabel='Stimulus', ylabel='Stimulus')

S_all = compute_stimulus_similarity(F)
classes = sorted(np.unique(all_targets))
intra_vals, inter_vals = [], []
for ci, cl in enumerate(classes):
    mask  = all_targets == cl
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for cl2 in classes[ci + 1:]:
        inter_vals.extend(S_all[np.ix_(mask, all_targets == cl2)].ravel())
intra_arr, inter_arr = np.array(intra_vals), np.array(inter_vals)
axes[1].hist(intra_arr, bins=60, alpha=0.6, density=True,
             label=f'Intra ({intra_arr.mean():.3f})')
axes[1].hist(inter_arr, bins=60, alpha=0.6, density=True,
             label=f'Inter ({inter_arr.mean():.3f})')
axes[1].set(xlabel='Cosine similarity', ylabel='Density',
            title='Intra- vs inter-class fingerprint similarity')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'nnls_fingerprints.pdf'), bbox_inches='tight')
plt.show()

# ── Plot 7: Embedding comparison ───────────────────────────────────────────────
# Panels: PCA(fingerprints) | PCA(last-layer) | PCA(all-layers) | MDS(fingerprints)
full_acts_id = np.concatenate([li[viz_idx] for li in layer_inputs], axis=1)
fig = plot_embedding_comparison(
    F[viz_idx], layer_inputs[-1][viz_idx], targets_viz,
    CLASS_NAMES, digit_targets=all_digits[viz_idx],
    activations_all=full_acts_id,
    title='ID test-set fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_id.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

### 4d — Near-OOD: excluded MNIST digits (2,5,6,7,8,9)

In [ ]:
OOD_DIGITS = [d for d in range(10) if d not in digit_filter]
print(f'OOD digits: {OOD_DIGITS}')

mnist_test_full = datasets.MNIST('../data/', train=False, download=True, transform=ToTensor())
ood_labels_all  = np.array(mnist_test_full.targets)
ood_indices     = np.where(np.isin(ood_labels_all, OOD_DIGITS))[0]
ood_subset      = Subset(mnist_test_full, ood_indices)

data_ood = collect_layer_inputs_generic(
    model, ood_subset, label_transform=label_transform_even_odd,
    only_correct=False, device=DEVICE,
)
ood_images  = data_ood['images']
ood_digits  = data_ood['digits']
ood_targets = data_ood['targets']
ood_inputs  = data_ood['layer_inputs']
n_ood       = len(ood_images)
print(f'OOD samples: {n_ood}')

projected_ood    = project_stimuli_onto_tree(nnls_root, ood_inputs)
factor_nodes_ood = extract_factor_tree_nodes(projected_ood)

# ── Factor tree per OOD digit ─────────────────────────────────────────────────
unique_ood = sorted(np.unique(ood_digits))
ncols = min(3, len(unique_ood))
nrows = int(np.ceil(len(unique_ood) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
for i, d in enumerate(unique_ood):
    ax   = axes[i // ncols][i % ncols]
    idx  = np.where(ood_digits == d)[0]
    acts = compute_factor_activations(factor_nodes_ood, idx)
    parity = 'even' if d % 2 == 0 else 'odd'
    plot_factor_tree(factor_nodes_ood, acts, ax=ax,
                     title=f'OOD digit {d} ({parity})  n={len(idx)}')
for j in range(len(unique_ood), nrows * ncols):
    axes[j // ncols][j % ncols].set_visible(False)
plt.suptitle('Near-OOD — Factor tree per excluded digit', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'near_ood_tree_per_digit.pdf'), bbox_inches='tight')
plt.show()

# ── Plot 7: Embedding comparison — ID test vs near-OOD ────────────────────────
N_EACH = 150
rng_m  = np.random.default_rng(7)
id_sub  = rng_m.choice(n_samples, min(N_EACH, n_samples), replace=False)
ood_sub = rng_m.choice(n_ood, min(N_EACH, n_ood), replace=False)

F_id_sub  = extract_fingerprint_matrix(nnls_root, id_sub)
F_ood_sub = extract_fingerprint_matrix(projected_ood, ood_sub)
F_combo   = np.concatenate([F_id_sub, F_ood_sub])
act_combo = np.concatenate([layer_inputs[-1][id_sub], ood_inputs[-1][ood_sub]])
lbl_combo = np.concatenate([all_targets[id_sub], ood_targets[ood_sub]])
digit_combo = np.concatenate([all_digits[id_sub], ood_digits[ood_sub]])
cond_combo  = (['ID'] * len(id_sub)) + (['near-OOD'] * len(ood_sub))

full_id_sub  = np.concatenate([li[id_sub]  for li in layer_inputs], axis=1)
full_ood_sub = np.concatenate([li[ood_sub] for li in ood_inputs],   axis=1)
full_combo   = np.concatenate([full_id_sub, full_ood_sub],           axis=0)

fig = plot_embedding_comparison(
    F_combo, act_combo, lbl_combo,
    CLASS_NAMES, digit_targets=digit_combo,
    condition_labels=cond_combo,
    activations_all=full_combo,
    title='Near-OOD vs ID fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_near_ood.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

### 4e — Far OOD: synthetic images

In [ ]:
rng_f = np.random.default_rng(99)
_chk  = (np.indices((IMAGE_SIDE, IMAGE_SIDE)).sum(0) % 2).astype(np.float32)

far_ood_arrays = {
    'gaussian_noise': np.clip(
        rng_f.normal(0.5, 0.25, (N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE)).astype(np.float32), 0, 1),
    'uniform_gray':   np.full((N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE), 0.5, dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk, (N_FAR_OOD, 1, IMAGE_SIDE, IMAGE_SIDE)).copy().astype(np.float32),
    'inverted_test':  np.clip(1.0 - all_images[:N_FAR_OOD], 0, 1).astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds = TensorDataset(torch.from_numpy(imgs),
                       torch.zeros(len(imgs), dtype=torch.long))
    d = collect_layer_inputs_generic(
        model, ds, label_transform=label_transform_even_odd,
        only_correct=False, device=DEVICE,
    )
    d['projected_root'] = project_stimuli_onto_tree(nnls_root, d['layer_inputs'])
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  model acc={(d["preds"] == d["targets"]).mean():.3f}')

# ── Factor tree per far-OOD type ──────────────────────────────────────────────
n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5), squeeze=False)
axes = axes[0]
for ax, (name, d) in zip(axes, far_ood_data.items()):
    acts = compute_factor_activations(d['factor_nodes'], np.arange(len(d['images'])))
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far OOD — factor tree activation per synthetic type', y=1.02, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'far_ood_factor_tree.pdf'), bbox_inches='tight')
plt.show()

# ── Plot 7: Embedding comparison — ID / near-OOD / far-OOD ───────────────────
N_MDS = 80
rng_m2 = np.random.default_rng(5)

F_parts, act_parts, full_parts, lbl_parts, digit_parts, cond_parts = [], [], [], [], [], []

# ID test
id_s2 = rng_m2.choice(n_samples, min(N_MDS, n_samples), replace=False)
F_parts.append(extract_fingerprint_matrix(nnls_root, id_s2))
act_parts.append(layer_inputs[-1][id_s2])
full_parts.append(np.concatenate([li[id_s2] for li in layer_inputs], axis=1))
lbl_parts.append(all_targets[id_s2])
digit_parts.append(all_digits[id_s2])
cond_parts.extend(['ID-test'] * len(id_s2))

# Near-OOD
ood_s2 = rng_m2.choice(n_ood, min(N_MDS, n_ood), replace=False)
F_parts.append(extract_fingerprint_matrix(projected_ood, ood_s2))
act_parts.append(ood_inputs[-1][ood_s2])
full_parts.append(np.concatenate([li[ood_s2] for li in ood_inputs], axis=1))
lbl_parts.append(ood_targets[ood_s2])
digit_parts.append(ood_digits[ood_s2])
cond_parts.extend(['near-OOD'] * len(ood_s2))

# Far-OOD types
for name, d in far_ood_data.items():
    n  = min(N_MDS, len(d['images']))
    ss = rng_m2.choice(len(d['images']), n, replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], ss))
    act_parts.append(d['layer_inputs'][-1][ss])
    full_parts.append(np.concatenate([li[ss] for li in d['layer_inputs']], axis=1))
    lbl_parts.append(d['targets'][ss])
    digit_parts.append(np.full(n, -1))
    cond_parts.extend([name] * n)

F_joint     = np.concatenate(F_parts)
act_joint   = np.concatenate(act_parts)
lbl_joint   = np.concatenate(lbl_parts)
digit_joint = np.concatenate(digit_parts)

fig = plot_embedding_comparison(
    F_joint, act_joint, lbl_joint,
    CLASS_NAMES, digit_targets=None,
    condition_labels=cond_parts,
    far_ood_conditions=list(far_ood_data.keys()),
    activations_all=np.concatenate(full_parts, axis=0),
    title='ID / near-OOD / far-OOD fingerprint embeddings',
)
fig.savefig(os.path.join(FIG_DIR, 'embedding_comparison_all_ood.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

In [ ]:
# ── Plot 5b: Model-seed robustness ────────────────────────────────────────────
print('\nModel-seed robustness: comparing root img_factors across 5 seeds...')
from src.robustness_utils import align_factors as _align_factors

_seed_root_factors = []
for seed_idx in range(N_SEEDS):
    m_s, _ = models_by_seed[seed_idx]

    _r = bft(m_s, val_loader, k_max=K_MAX_PER_LAYER, n_branches=[1,1,N_CLASSES],
                stimulus_threshold=STIMULUS_THRESHOLD, n_jobs=3)

    H = _r.root.img_factors          # (N, K_root)
    norms = np.linalg.norm(H, axis=0, keepdims=True)
    _seed_root_factors.append(H / (norms + 1e-12))

ref = _seed_root_factors[0]
others = _seed_root_factors[1:]
_, sim_scores = _align_factors(ref, others)
print(f'Root factor alignment across seeds: {np.array(sim_scores).mean():.3f} ± {np.array(sim_scores).std():.3f}')
